# EX 1 -  Image Classification Using convolution kernel: classify cats or dogs from images

- Build the model
    - Model consists of three convolution blocks with a max pool layer in each of them. There's a fully connected layer with 512 units on top of it that is activated by a `relu` activation function.
    - Choose the ADAM optimizer and binary cross entropy loss function.
- Train the model
    - Parameters for pre-processing the dataset and training the network: batch_size = 128, epochs = 15, image height = 150, image width = 150 

- [Hints] 
    - Builds an image classifier using a `tf.keras.Sequential` model and load data using `tf.keras.preprocessing.image.ImageDataGenerator`. 
    - Building _data input pipelines_ using the `tf.keras.preprocessing.image.ImageDataGenerator` class to efficiently work with data to use with the model.

# Data Preprocessing and Exploration
---

## Step 1: Download and Extract the Dataset

**Objective:**  
Download and extract the Cats vs Dogs dataset from the provided URL so that the training and validation directories are accessible.

**Instructions:**
- Use `tf.keras.utils.get_file` to download the dataset archive from the given URL.
- Extract the archive into a local directory.
- Write a function that searches through the extracted files to locate the main dataset folder containing the `train` subdirectory.
- Print the dataset directory to confirm that the extraction and search were successful.

---

## Step 2: Set Up Data Directories and Count Images

**Objective:**  
Define the file paths for the training and validation sets, and verify the dataset by counting the images.

**Instructions:**
- Assign variables for `train_dir` and `validation_dir` using `os.path.join` with the dataset folder.
- Define subdirectories for cats and dogs within both the training and validation directories.
- Use `os.listdir` to count the number of images in each subdirectory.
- Compute and print the total number of training images and validation images to ensure that the dataset structure is correct.

---

## Step 3: Configure Image Data Generators and Visualize a Batch

**Objective:**  
Pre-process the images for model training and verify the data loading process by visualizing sample images.

**Instructions:**
- Define the pre-processing parameters: `batch_size`, `IMG_HEIGHT`, `IMG_WIDTH`, and `epochs`.
- Create two instances of `ImageDataGenerator` (one for training and one for validation) and apply rescaling (e.g., `rescale=1./255`).
- Use the `flow_from_directory` method to load and resize images from the training and validation directories.
- Retrieve a batch of images using the `next` function on the training data generator.
- Write a helper function that plots a grid (e.g., 1 row with 5 columns) of sample images, and call it to verify that the images are correctly pre-processed.


In [ ]:
# Author: Francesco Esposito (fe.digi@cbs.dk)
# Src TensorFlow.org
# Import packages
import os
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import tensorflow as tf


from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Conv2D, Flatten, Dropout, MaxPooling2D
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import accuracy_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from tensorflow.keras import layers, losses
from tensorflow.keras.datasets import fashion_mnist
from tensorflow.keras.models import Model

In [ ]:
# Load data: from <a href="https://www.kaggle.com/c/dogs-vs-cats/data" target="_blank">Dogs vs Cats</a> dataset from Kaggle. 
# Download the archive version of the dataset and store it in the "/tmp/" directory.

_URL = 'https://storage.googleapis.com/mledu-datasets/cats_and_dogs_filtered.zip'

path_to_zip = tf.keras.utils.get_file('cats_and_dogs.zip', origin=_URL, extract=True)

PATH = os.path.join(os.path.dirname(path_to_zip), 'cats_and_dogs_filtered')

In [2]:
# If the code does not work, try the following one:

#_URL = 'https://storage.googleapis.com/mledu-datasets/cats_and_dogs_filtered.zip'
#path_to_zip = tf.keras.utils.get_file('cats_and_dogs.zip', origin=_URL, extract=True)
#base_dir = os.path.dirname(path_to_zip)

#def find_dataset_folder(root_dir):
    #for current_dir, subdirs, files in os.walk(root_dir):
        #if 'train' in subdirs:
            #return current_dir
    #return None

#PATH = find_dataset_folder(base_dir)
#if PATH is None:
    #raise Exception("Could not find the dataset folder with a 'train' subdirectory.")
#else:
    #print("Dataset directory found:", PATH)

In [ ]:
# [TODO] After extracting its contents, assign variables with the proper file path for the training and validation set.
# 1. Identify the main dataset directory stored in the variable 'PATH'.
# 2. Use 'os.path.join' to append the 'train' folder name to 'PATH' and assign it to the variable.
# 3. Repeat the same process for the 'validation' folder and assign it to the variable.
# 4. Print the variables to confirm they point to the correct directories.

In [ ]:
# 5. Assume you have a variable representing the base directory for your training images (for example, <your_training_directory>).
# 6. Use os.path.join to append the folder name for one category (e.g., 'cats') to your training base directory, and store this in a variable.
# 7. Repeat the process for another category (e.g., 'dogs') for the training set.
# 8. Similarly, assume you have a variable for your validation base directory (for example, <your_validation_directory>).
# 9. Use os.path.join to construct paths for each category in the validation set.
# Note: The variable names you choose are arbitrary; just be sure to use them consistently.

In [ ]:
# [TODO] Understand the data: Look at how many cats and dogs images are in the training and validation directory:
# Please adapt to the variables you defined in the previous step
num_cats_tr = len(os.listdir(train_cats_dir))
num_dogs_tr = len(os.listdir(train_dogs_dir))

num_cats_val = len(os.listdir(validation_cats_dir))
num_dogs_val = len(os.listdir(validation_dogs_dir))

total_train = num_cats_tr + num_dogs_tr
total_val = num_cats_val + num_dogs_val


print('total training cat images:', num_cats_tr)
print('total training dog images:', num_dogs_tr)

print('total validation cat images:', num_cats_val)
print('total validation dog images:', num_dogs_val)
print("--")
print("Total training images:", total_train)
print("Total validation images:", total_val)

In [ ]:
# [TODO], set up variables to use while pre-processing the dataset and training the network.
# 1. Set 'batch_size' to determine how many images to process in one iteration.
# 2. Set 'epochs' to define the number of times the model will see the entire training dataset.
# 3. Define 'IMG_HEIGHT' and 'IMG_WIDTH' to resize all images to a standard size.

# Model Building

## Step 1: Define the CNN Architecture

**Objective:**  
Construct a CNN model that can classify images into two categories (cats and dogs).

**Instructions:**
- Import the required layers from TensorFlow Keras, such as `Conv2D`, `MaxPooling2D`, `Flatten`, and `Dense`.
- Initialize a sequential model.
- Add a convolutional layer with a specified number of filters (e.g., 16), a kernel size (e.g., 3), and use `relu` activation with `padding='same'`. Make sure to define the input shape using `IMG_HEIGHT`, `IMG_WIDTH`, and the number of channels (3 for RGB).
- Follow the convolutional layer with a max pooling layer.
- Add additional convolutional and max pooling layers (for example, with 32 and then 64 filters) to deepen the network.
- Flatten the output from the convolutional layers to prepare it for the dense layers.
- Add a dense layer with a substantial number of neurons (e.g., 512) and `relu` activation.
- Add a final dense layer with a single unit for binary classification.

---

## Step 2: Compile the Model

**Objective:**  
Prepare the model for training by specifying the optimizer, loss function, and evaluation metrics.

**Instructions:**
- Compile the model using the ADAM optimizer.
- Specify the loss function as binary crossentropy (using `tf.keras.losses.BinaryCrossentropy` with `from_logits=True` if applicable).
- Include `accuracy` in the metrics.
- Use the model's `summary` method to display the network architecture and verify that all layers have been correctly configured.


In [ ]:
# Data preparation: Format the images into appropriately pre-processed floating point tensors before feeding to the network:

# 1. Read images from the disk.
# 2. Decode the image contents to form a proper grid (based on RGB values).
# 3. Convert the images into floating point tensors.
# 4. Rescale the tensor values from [0, 255] to [0, 1] for better neural network performance.
# 5. Use the ImageDataGenerator class from tf.keras to perform all the above steps.

In [ ]:
# [TODO] After defining the generators for training and validation images, the `flow_from_directory` method load images 
# from the disk, applies rescaling, and resizes the images into the required dimensions.
# 1. Use the flow_from_directory method on the training ImageDataGenerator to:
#    - Read images from the training directory.
#    - Shuffle the images.
#    - Resize them to (IMG_HEIGHT, IMG_WIDTH).
#    - Set the class mode to 'binary' for binary classification.
# 2. Repeat the process for the validation set (no need for shuffling).

In [ ]:
# [TODO] Visualize the training images by extracting a batch of images from the training 
# generator—which is 32 images in this example—then plot five of them with `matplotlib`.

# [Hints] The `next` function returns a batch from the dataset. 
# The return value of `next` function is in form of `(x_train, y_train)` 
# Discard the labels to only visualize the training images.


In [ ]:
# [TODO] Plot images in the form of a grid with 1 row and 5 columns where images are placed in each column
# 1. Define a function (e.g., plotImages) that takes an array of images as input.
# 2. Inside the function, create a subplot grid with 1 row and 5 columns using plt.subplots.
# 3. Flatten the axes array to simplify iterating over the subplots.
# 4. Loop through the first 5 images in the array, display each image using ax.imshow, and disable the axis using ax.axis('off') for a cleaner look.
# 5. Adjust the layout with plt.tight_layout() and display the plot using plt.show().

In [ ]:
# [TODO] Create the model

# 1. Initialize a Sequential model.
# 2. Add a Conv2D layer with 16 filters, kernel size 3, 'same' padding, and 'relu' activation.
#    - Specify the input shape as (IMG_HEIGHT, IMG_WIDTH, 3) for RGB images.
# 3. Add a MaxPooling2D layer to reduce the spatial dimensions.
# 4. Add a second Conv2D layer with 32 filters, kernel size 3, 'same' padding, and 'relu' activation.
# 5. Add another MaxPooling2D layer.
# 6. Add a third Conv2D layer with 64 filters, kernel size 3, 'same' padding, and 'relu' activation.
# 7. Add a third MaxPooling2D layer.
# 8. Flatten the output of the convolutional layers to convert it into a 1D vector.
# 9. Add a Dense layer with 512 neurons and 'relu' activation.
# 10. Add a final Dense layer with 1 neuron for binary classification output.

In [ ]:
# [TODO] Compile the model: *ADAM* optimizer and *binary cross entropy* loss function. 
# view training and validation accuracy for each training epoch, pass the `metrics` argument.
# 1. Set the optimizer to 'adam' for adaptive learning.
# 2. Use binary crossentropy as the loss function, suitable for binary classification tasks.
# 3. Include 'accuracy' in the metrics to monitor both training and validation performance during each epoch.

In [ ]:
# [TODO] View all the layers of the network using the model's `summary` method

# Model Training and Evaluation

## Step 1: Train the Model

**Objective:**  
Fit the CNN model using the training data while monitoring its performance on the validation set.

**Instructions:**
- Train the model using the `fit` method, providing the training data generator as input.
- Set the number of epochs and calculate steps per epoch based on the total number of training images and the batch size.
- Provide the validation data generator to the `fit` method to monitor validation performance during training.
- Save the training history, which contains metrics such as loss and accuracy for both training and validation.

---

## Step 2: Visualize Training and Validation Metrics

**Objective:**  
Assess the performance of the model by plotting the training and validation accuracy and loss over epochs.

**Instructions:**
- Extract the training and validation accuracy and loss values from the history object returned by the `fit` method.
- Create an array representing the range of epochs for plotting.
- Using Matplotlib, plot the training accuracy and validation accuracy in one subplot with proper labels and legends.
- In a separate subplot, plot the training loss and validation loss with clear labeling and legends.
- Analyze the plots to determine the model’s learning progress, and discuss potential signs of overfitting or underfitting.


In [ ]:
# [TODO] Use the `fit_generator` method of the `ImageDataGenerator` class to train the network.
# 1. Use the model's fit() method to train the network with the training data generator (train_data_gen).
# 2. Set steps_per_epoch to total_train // batch_size so that the model processes all training images in each epoch.
# 3. Define the number of epochs with the 'epochs' variable.
# 4. Provide the validation data generator (val_data_gen) as validation_data to monitor performance.
# 5. Set validation_steps to total_val // batch_size to cover all validation images.
# 6. Store the training history in a variable (e.g., history) for later evaluation and visualization.

In [ ]:
# [TODO] Now visualize the results after training the network
# Please adapt based on the variables you defined
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']

loss=history.history['loss']
val_loss=history.history['val_loss']

epochs_range = range(epochs)

# Plot the figure for Training and Validation Accuracy
plt.figure(figsize=(8, 8))
plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label='Training Accuracy')
plt.plot(epochs_range, val_acc, label='Validation Accuracy')
plt.legend(loc='lower right')
plt.title('Training and Validation Accuracy')

# Plot the figure for Training and Validation Loss
plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label='Training Loss')
plt.plot(epochs_range, val_loss, label='Validation Loss')
plt.legend(loc='upper right')
plt.title('Training and Validation Loss')
plt.show()

# EX 2 - Autoencoder for Fashion MNIST Denoising

In this part, we will build and train a basic autoencoder to denoise images from the Fashion MNIST dataset. Follow the steps below:

## Step 1: Load and Preprocess the Data
- **Load the Dataset:**  
  Use `fashion_mnist.load_data()` to load the Fashion MNIST dataset.  
- **Normalize the Images:**  
  Convert the images to `float32` and scale the pixel values to the range [0, 1] by dividing by 255.  
- **Verify Data Shapes:**  
  Print the shapes of the training and test sets to confirm that each image is 28x28 pixels.

## Step 2: Expand Dimensions for Conv2D Compatibility
- **Add a Channel Dimension:**  
  Expand the dimensions of `x_train` and `x_test` using `np.expand_dims()` so that the images have shape (28, 28, 1), which is required by Conv2D layers.

## Step 3: Add Noise to the Images
- **Define a Noise Factor:**  
  Choose a noise factor (e.g., 0.5) to control the amount of noise added.  
- **Create Noisy Versions:**  
  Add Gaussian noise to the training and test images using `np.random.normal()` and the noise factor, resulting in `x_train_noisy` and `x_test_noisy`.  
- **Clip the Values:**  
  Use `np.clip()` to ensure that the pixel values of the noisy images remain between 0 and 1.  
- **Verify Consistency:**  
  Print the shapes of the noisy images to ensure they match the original dimensions.

## Step 4: Build the Autoencoder Model
- **Define a Custom Model:**  
  Create a custom autoencoder class (e.g., `Denoise`) that inherits from `tf.keras.Model`.  
- **Build the Encoder:**  
  In the class’s `__init__` method, build the encoder as a Sequential model:
  - Add an Input layer for images of shape (28, 28, 1).
  - Add a Conv2D layer with 16 filters, kernel size (3, 3), `relu` activation, `same` padding, and strides=2.
  - Add a second Conv2D layer with 8 filters (using similar parameters) to further downsample the images.
- **Build the Decoder:**  
  Also in the `__init__` method, build the decoder as a Sequential model:
  - Use Conv2DTranspose layers to upsample the encoded feature maps.
  - End with a Conv2D layer with 1 filter, kernel size (3, 3), `sigmoid` activation, and `same` padding to reconstruct the image.
- **Define the Call Method:**  
  Implement the `call()` method to process the input through the encoder and then the decoder, returning the reconstructed image.

## Step 5: Compile and Train the Autoencoder
- **Instantiate the Model:**  
  Create an instance of your autoencoder.
- **Compile the Model:**  
  Use the `adam` optimizer and Mean Squared Error (MSE) loss function to compile the model.
- **Train the Model:**  
  Train the autoencoder using `autoencoder.fit()`, with:
  - `x_train_noisy` as the input and `x_train` as the target,
  - A chosen number of epochs (e.g., 10),
  - Shuffling enabled,
  - And using `(x_test_noisy, x_test)` as the validation data.

## Step 6: Evaluate and Visualize the Results
- **Model Summaries:**  
  Print summaries of both the encoder and decoder to observe how the image dimensions are reduced (e.g., from 28x28 to 7x7) and then reconstructed back to 28x28.
- **Generate Reconstructions:**  
  Use the encoder to encode the test images and then pass the encoded representations through the decoder to obtain denoised (reconstructed) images.
- **Visualize the Outcome:**  
  Create a visualization using matplotlib:
  - Set up a figure with two rows and a fixed number of columns (e.g., 10 samples).
  - In the top row, display the original noisy test images labeled "original + noise".
  - In the bottom row, display the corresponding reconstructed images labeled "reconstructed".
  - Remove axis ticks and labels for clarity.


In [ ]:
# Load the dataset 
# Train the basic autoencoder using the Fashon MNIST dataset. 
# Each image in this dataset is 28x28 pixels.

(x_train, _), (x_test, _) = fashion_mnist.load_data()

x_train = x_train.astype('float32') / 255.
x_test = x_test.astype('float32') / 255.

In [ ]:
# Expand dimensions to add the channel (needed for Conv2D)
x_train = np.expand_dims(x_train, -1)
x_test = np.expand_dims(x_test, -1)

# Add noise to the images
noise_factor = 0.5
x_train_noisy = x_train + noise_factor * np.random.normal(loc=0.0, scale=1.0, size=x_train.shape)
x_test_noisy = x_test + noise_factor * np.random.normal(loc=0.0, scale=1.0, size=x_test.shape)
x_train_noisy = np.clip(x_train_noisy, 0., 1.)
x_test_noisy = np.clip(x_test_noisy, 0., 1.)

print(x_train.shape)
print(x_test.shape)

In [ ]:
# Step 1: Create a custom autoencoder model by subclassing tf.keras.Model.
# This model will learn to remove noise from images.

# Step 2: In the __init__ method, call the parent class constructor with super().

# Step 3: Build the encoder part of the autoencoder.
# - Use a Sequential model.
# - Start with an Input layer for images of shape (28, 28, 1).
# - Add a Conv2D layer with 16 filters, kernel size (3, 3), 'relu' activation, 'same' padding, and strides=2 for downsampling.
# - Add a second Conv2D layer with 8 filters (with the same parameters) to further downsample the images.

# Step 4: Build the decoder part of the autoencoder.
# - Use another Sequential model.
# - Add a Conv2DTranspose layer with 8 filters, kernel size 3, strides=2, 'relu' activation, and 'same' padding to upsample.
# - Add a second Conv2DTranspose layer with 16 filters to further upsample.
# - End with a Conv2D layer with 1 filter, kernel size (3, 3), 'sigmoid' activation, and 'same' padding to reconstruct the image.

# Step 5: Define the call() method.
# - Pass the input through the encoder to obtain the encoded representation.
# - Pass the encoded representation through the decoder to reconstruct the image.
# - Return the reconstructed image.

# Step 6: Instantiate the autoencoder model by creating an object of the custom class.

In [ ]:
# Compile the Autoencoder Model
# - Use the Adam optimizer for adaptive learning.
# - Set the loss function to Mean Squared Error (MSE) since it's appropriate for reconstruction tasks in autoencoders.

In [ ]:
autoencoder.fit(x_train_noisy, x_train,
                epochs=10,
                shuffle=True,
                validation_data=(x_test_noisy, x_test))

In [ ]:
# Look at a summary of the encoder. Notice how the images are downsampled from 28x28 to 7x7.
autoencoder.encoder.summary()

In [ ]:
# The decoder upsamples the images back from 7x7 to 28x28.
autoencoder.decoder.summary()

In [ ]:
# Plotting both the noisy images and the denoised images produced by the autoencoder.
encoded_imgs = autoencoder.encoder(x_test).numpy()
decoded_imgs = autoencoder.decoder(encoded_imgs).numpy()

In [ ]:
n = 10
plt.figure(figsize=(20, 4))
for i in range(n):

    # display original + noise
    ax = plt.subplot(2, n, i + 1)
    plt.title("original + noise")
    plt.imshow(tf.squeeze(x_test_noisy[i]))
    plt.gray()
    ax.get_xaxis().set_visible(False)
    ax.get_yaxis().set_visible(False)

    # display reconstruction
    bx = plt.subplot(2, n, i + n + 1)
    plt.title("reconstructed")
    plt.imshow(tf.squeeze(decoded_imgs[i]))
    plt.gray()
    bx.get_xaxis().set_visible(False)
    bx.get_yaxis().set_visible(False)
plt.show()